# 07 — Predictive model: the final figures

**Figure 4** (one figure, two halves) and **Figure 5**, and the checks that go with them.
First half: Cam-CAN, cross-validated. Second half: trained on all of Cam-CAN, tested on HCP.

The slow parts (the SHAP refits, the transfer's fits) are **read from cache**: run
`sbatch slurm/predictive_cache.sbatch` once (with the same `SPACE` / `MODEL` / `PICK` as here when
they are not the defaults); a part not cached is computed here, once, and cached.

| plot | was | what |
|---|---|---|
| Figure 4, left | 10.9 | bstm vs static FC, Cam-CAN (80% / 20%), what the bstm model uses |
| 7.1 | 10.7 | volcano, anxiety / depression vs Normal |
| Figure 5 | 10.12 | where the model looks, Cam-CAN |
| 7.2, 7.3 | 10.10, 10.11 | observed vs out-of-fold prediction; stability over fold seeds |
| Figure 4, right | 13.4 | bstm vs static FC, Cam-CAN → HCP |
| 7.4 | 13.5 | predicted vs observed, HCP |


## Part 1 — Cam-CAN, cross-validated

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from fmri_decomposition.supervised_model import hads as ht
from fmri_decomposition.supervised_model import report as rp
from fmri_decomposition.supervised_model import transfer_tables as tt
from fmri_decomposition.figures import hads as hp

# Cam-CAN, notebook 10's settings (report.CAM); override any of them here, e.g. save_dir="figures/07"
cfg = rp.cam_settings()
st = ht.Study(cfg)
T0, T1 = cfg.tasks
print(f"phenotype: {len(st.pheno):,} subjects;",
      ", ".join(f"{lab} {st.pheno[lab].value_counts().sort_index().astype(int).to_dict()}" for lab in cfg.targets))


In [ ]:
SPACE = None        # None = the best state set by CV; e.g. dict(atlas="harvardoxford", window_s=30.0, states="HMM2_umap3_8")
MODEL = None        # None = the best model of graph_arms; e.g. dict(model="hgb", arm="all [cond]")
base, options, space, model = rp.cam_choices(st, SPACE, MODEL)
label = f"{model['model']} · {model['arm']} · {ht.set_label(space)}"
print("space:", ht.set_label(space), "| model:", label)


### Figure 4, left half (was 10.9)

Drawn here, shown with the right half at the end.

In [ ]:
# Figure 4, left half (was 10.9)
E1_MODEL = "hgb"                    # or "ridge"
FC_ARM = "edges"                    # or "global"
FC_ATLAS = "yeo7"                   # static FC: "yeo7" (main figure) | None = the states' atlas
perf = st.performance_trees(space, model["arm"], E1_MODEL, base, fc_arm=FC_ARM, fc_atlas=FC_ATLAS)
e1_seeds = st.tree_seeds(space, model["arm"], E1_MODEL, fc_arm=FC_ARM, fc_atlas=FC_ATLAS)
e1_shares = st.block_shares(st.deep(space, dict(model=E1_MODEL, arm=model["arm"])))
_fc_atlas = FC_ATLAS or space["atlas"]
fc_note = "" if (perf["tree"] == "fcm").any() else f" -- no fcm run on {_fc_atlas}"
fig4_left = hp.fig_performance(perf, cfg, "",
                               {"bstm": f"selected: {ht.set_label(space)} · {model['arm']}",
                                "fcm": f"{_fc_atlas} · same model · {FC_ARM}{fc_note}"},
                               E1_MODEL, seeds=e1_seeds, shares=e1_shares, show=False)


### Plot 7.1 — anxiety and depression vs Normal (was 10.7)

In [ ]:
# Plot 7.1
grp = st.group_tests(space)
hp.fig_volcanos(grp, cfg, f"{cfg.case_from}+ vs Normal (on both scales), adjusted · {ht.set_label(space)}",
                "D2_volcano_groups", f"log2 fold change  ({cfg.case_from}+ vs Normal, adjusted, shrink={cfg.shrink:g})",
                alpha=0.10, fc_line=0.25, xlim=0)


### Figure 5 — where the model looks, Cam-CAN (was 10.12)

In [ ]:
# Figure 5
paired = st.paired(space)              # movie vs rest, Normal group (the dynamics beside the graphs)
pooled = st.pooled(space)
dyn = st.dynamics_marks(space, paired, grp)
deep = st.deep(space, model)           # cached by slurm/predictive_cache.sbatch
st.check(deep)
mmarks = st.model_marks(space, deep, pooled)
refs_model = {(t, lab): (t, "Normal") for t in cfg.tasks for lab in cfg.targets}
hp.fig_graph(pooled, mmarks, cfg, refs_model, {k: hp.RED for k in refs_model}, f"Figure 5  where the model looks · {label}",
             [(hp.RED, f"red = the model's top {cfg.highlight_top} graph variables by mean |SHAP|, top in "
                       f"≥{cfg.stable_min:.0%} of seeds, |Cohen's d| ≥ {cfg.min_d} vs Normal; label = change in pp")],
             "figure5_graph", dyn=dyn.assign(hit=False))


### Plots 7.2, 7.3 — prediction and its stability (were 10.10, 10.11)

In [ ]:
# Plots 7.2, 7.3
hp.fig_predicted(deep, cfg, f"observed category vs out-of-fold prediction · {label}")
hp.fig_stability(deep, cfg, f"stability over fold seeds · {label}")


## Part 2 — trained on all of Cam-CAN, tested on HCP

In [ ]:
# Cam-CAN -> HCP, notebook 13's setup (report.transfer_setup)
TARGET = "category"      # "raw" | "category" (the manuscript)
train, test, train_cfg, test_cfg, ROOT = rp.transfer_setup(TARGET)
raw_rows, rows_t = rp.transferred_rows(ROOT, TARGET)
print(f"{len(rows_t)} transferred configuration(s) over {rows_t['states'].nunique()} state set(s); rows from {ROOT}")


In [ ]:
PICK = {}          # narrow by any column of rows_t, e.g. {"model": "hgb"}, {"states": "HMM2_umap3_8"}
RANK = 0           # 0 = the best (interval clearing the baseline first), 1 = the runner-up, ...
best, ranked, n_clear = rp.choose_best(rows_t, PICK, RANK)
display(ranked.head(10)[["states", "atlas", "window", "arm", "model", "task", "phenotype", "score", "lo", "hi", "base_score"]])
space_t = rp._space(best)
print(f"{n_clear} configuration(s) clear their baseline; chosen: {best['label']} · {best['arm']} · {best['model']} · "
      f"{best['task']} · {best['phenotype']}  score {best['score']:+.3f} [{best['lo']:+.3f}, {best['hi']:+.3f}]")


### Figure 4, right half (was 13.4)

In [ ]:
# Figure 4, right half (was 13.4)
FC_ATLAS_T = "yeo7"
perf_t = tt.transfer_perf(rows_t, best, fc_arm=train_cfg.fc_arm, fc_atlas=FC_ATLAS_T)
shares_t = rp.transfer_shares_cached(train, test, space_t, best["arm"], best["model"])   # cached
fig4_right = hp.fig_performance(
    perf_t, train_cfg, "",
    {"bstm": f"{best['label']} · {best['arm']}", "fcm": f"{FC_ATLAS_T or best['atlas']} · {train_cfg.fc_arm}"},
    best["model"], shares=shares_t,
    share_label="share of mean |SHAP| on the TEST subjects (one fit on all of Cam-CAN, no folds)",
    show=False)


### Plot 7.4 — predicted vs observed, HCP (was 13.5)

In [ ]:
# Plot 7.4
pred = rp.transfer_predictions_cached(train, test, best, fc_arm=train_cfg.fc_arm)   # cached
hp.fig_scatter(pred, train_cfg, title=f"predicted vs observed · {best['label']} · {best['model']}")


## Figure 4

In [ ]:
# Figure 4
hp.fig_figure4(fig4_left, fig4_right, cfg,
               titles=("Train in 80% Cam-CAN → test in 20% Cam-CAN",
                       "Train in 100% Cam-CAN → test in 100% HCP"));
